In [2]:
from datasets import load_dataset

ds = load_dataset("mamtakumar/seven-phishing-email-datasets")

README.md:   0%|          | 0.00/5.50k [00:00<?, ?B/s]

train.parquet: reconstructing file:   0%|          |  0.00B /  184MB            

train.parquet: downloading bytes:           |  0.00B            

test.parquet: reconstructing file:   0%|          |  0.00B / 22.7MB            

test.parquet: downloading bytes:           |  0.00B            

eval.parquet: reconstructing file:   0%|          |  0.00B / 22.7MB            

eval.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/162413 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/40604 [00:00<?, ? examples/s]

In [3]:
print(ds)

DatasetDict({
    train: Dataset({
        features: ['text', 'subject', 'label', 'sender', 'receiver', 'date', 'urls', 'dataset_name'],
        num_rows: 162413
    })
    test: Dataset({
        features: ['text', 'subject', 'label', 'sender', 'receiver', 'date', 'urls', 'dataset_name'],
        num_rows: 40604
    })
})


In [4]:
print("\nAvailable splits:", list(ds.keys()))


Available splits: ['train', 'test']


In [5]:
for split_name, split_data in ds.items():
    print(f"\nSplit: {split_name}")
    print("Rows:", len(split_data))
    print("Columns:", split_data.column_names)


Split: train
Rows: 162413
Columns: ['text', 'subject', 'label', 'sender', 'receiver', 'date', 'urls', 'dataset_name']

Split: test
Rows: 40604
Columns: ['text', 'subject', 'label', 'sender', 'receiver', 'date', 'urls', 'dataset_name']


In [6]:

from collections import Counter

train_data = ds["train"]

print("FEATURE SCHEMA")
print(train_data.features)

print("\nLABEL COUNTS")
print(Counter(train_data["label"]))

print("\nTHREE SAMPLE RECORDS")

for i in range(min(3, len(train_data))):
    record = train_data[i]

    print(f"\n--- Record {i} ---")
    for column in train_data.column_names:
        value = record[column]

        # Keep potentially long email text manageable.
        if column in ("text", "subject") and isinstance(value, str):
            value = value[:300]

        print(f"{column}: {value!r}")


FEATURE SCHEMA
{'text': Value('string'), 'subject': Value('string'), 'label': Value('int64'), 'sender': Value('string'), 'receiver': Value('string'), 'date': Value('timestamp[ns]'), 'urls': Value('int64'), 'dataset_name': Value('string')}

LABEL COUNTS
Counter({0: 86951, 1: 75462})

THREE SAMPLE RECORDS

--- Record 0 ---
text: "\n \n \n\n\nmeat helpful round An afterclang of Cowley's mouth chords closed, died on tdesire And Richie like debt Goulding drank his noise Power and Leopold Blsuccessful struck cough Libel action, says he, slow for ten thousand pounds.Very umbrella tame kind hat grip of you, says Bloom.\nQuavering the ma"
subject: 'When will it be okay'
label: 1
sender: 'Lissette Patterson <jstepanekov@eve-team.com>'
receiver: 'Olympia <dmason@plg2.math.uwaterloo.ca>'
date: Timestamp('2007-05-02 00:31:14')
urls: 0
dataset_name: 'TREC-07'

--- Record 1 ---
text: 'Avail to the E-store and avail to timely support.\n\nOrdering over 45 dollars, domestic customers are able to have ze

In [7]:

from collections import Counter

train_data = ds["train"]

print("Schema:")
print(train_data.features)

print("\nLabel counts:")
print(Counter(train_data["label"]))

print("\nSamples:")
for i in range(3):
    row = train_data[i]
    print({
        "label": row["label"],
        "subject": str(row["subject"])[:150],
        "text": str(row["text"])[:250],
        "dataset_name": row["dataset_name"],
    })


Schema:
{'text': Value('string'), 'subject': Value('string'), 'label': Value('int64'), 'sender': Value('string'), 'receiver': Value('string'), 'date': Value('timestamp[ns]'), 'urls': Value('int64'), 'dataset_name': Value('string')}

Label counts:
Counter({0: 86951, 1: 75462})

Samples:
{'label': 1, 'subject': 'When will it be okay', 'text': "\n \n \n\n\nmeat helpful round An afterclang of Cowley's mouth chords closed, died on tdesire And Richie like debt Goulding drank his noise Power and Leopold Blsuccessful struck cough Libel action, says he, slow for ten thousand pounds.Very umbrella tame ", 'dataset_name': 'TREC-07'}
{'label': 1, 'subject': 'Havea method to reduce the high expense of iink', 'text': 'Avail to the E-store and avail to timely support.\n\nOrdering over 45 dollars, domestic customers are able to have zero charged\ndelivery!\n\nOrdering over 45 bucks, domestic customers are able to have zero charged\ndelivery!\n\nIt is a better provider for i', 'dataset_name': 'TREC-05'}

In [8]:

import pandas as pd

train_df = ds["train"].to_pandas()

print("LABEL COUNTS BY SOURCE")
print(pd.crosstab(
    train_df["dataset_name"],
    train_df["label"],
    margins=True
))

print("\nMISSING VALUES")
print(train_df.isna().sum())

print("\nEMPTY EMAIL TEXT")
empty_text = (
    train_df["text"].isna()
    | train_df["text"].str.strip().eq("")
)
print(f"{empty_text.sum():,} empty or missing email bodies")

print("\nDUPLICATE TEXT")
print(
    "Exact duplicate bodies:",
    train_df["text"].duplicated().sum()
)

print("\nDATASET REVISION")
print("Record this separately from the dataset ID.")


LABEL COUNTS BY SOURCE
label             0      1     All
dataset_name                      
Assassin       3234   1351    4585
CEAS-08       13784  17366   31150
Enron         12643  11206   23849
Ling           1926    365    2291
TREC-05       26029  18432   44461
TREC-06        9904   3209   13113
TREC-07       19431  23533   42964
All           86951  75462  162413

MISSING VALUES
text                0
subject             0
label               0
sender          26140
receiver        26140
date            28898
urls            26140
dataset_name        0
dtype: int64

EMPTY EMAIL TEXT
4 empty or missing email bodies

DUPLICATE TEXT
Exact duplicate bodies: 9

DATASET REVISION
Record this separately from the dataset ID.


In [12]:

import requests

dataset_id = "mamtakumar/seven-phishing-email-datasets"

api_url = f"https://huggingface.co/api/datasets/{dataset_id}"
api_response = requests.get(api_url, timeout=30)
api_response.raise_for_status()
metadata = api_response.json()

print("Dataset revision:", metadata.get("sha"))
print("Declared license:", metadata.get("cardData", {}).get("license"))

readme_url = (
    f"https://huggingface.co/datasets/{dataset_id}/raw/main/README.md"
)
readme_response = requests.get(readme_url, timeout=30)
readme_response.raise_for_status()

lines = readme_response.text.splitlines()
keywords = ("label", "license", "source", "dataset", "phishing", "spam")

print("\nRelevant dataset-card sections:\n")

for i, line in enumerate(lines):
    if any(word in line.lower() for word in keywords):
        start = max(0, i - 1)
        end = min(len(lines), i + 2)
        print(f"Lines {start + 1}-{end}:")
        print("\n".join(lines[start:end]))
        print("-" * 40)


Dataset revision: c9440a2fd59c3992e53c361f6afbb4fa8952cd17
Declared license: other

Relevant dataset-card sections:

Lines 1-3:
---
pretty_name: Seven Phishing/Spam Email Datasets
language:
----------------------------------------
Lines 12-14:
  - 100K<n<1M
source_datasets:
  - original
----------------------------------------
Lines 16-18:
  - no-annotation
license: other
tags:
----------------------------------------
Lines 19-21:
  - email
  - phishing
  - spam
----------------------------------------
Lines 20-22:
  - phishing
  - spam
  - security
----------------------------------------
Lines 25-27:

# Dataset Card for Seven Phishing/Spam Email Datasets

----------------------------------------
Lines 27-29:

## Dataset Summary

----------------------------------------
Lines 29-31:

This dataset is a unified, row-level email corpus built from seven commonly used public email datasets. It is intended for research on phishing/spam detection and related email-text classification tasks.


In [13]:

import json
from pathlib import Path

audit = {
    "project": "MailSleuth AI",
    "dataset_id": "mamtakumar/seven-phishing-email-datasets",
    "dataset_revision": "c9440a2fd59c3992e53c361f6afbb4fa8952cd17",
    "target_column": "label",
    "documented_label_mapping": {
        "0": "benign/legitimate (ham)",
        "1": "phishing/spam; may include ordinary spam",
    },
    "sources": [
        "Assassin", "CEAS-08", "Enron", "Ling",
        "TREC-05", "TREC-06", "TREC-07",
    ],
    "train_rows": 162413,
    "test_rows": 40604,
    "train_label_counts": {
        "0": 86951,
        "1": 75462,
    },
    "train_empty_or_missing_text": 4,
    "train_exact_duplicate_bodies": 9,
    "declared_license": "other",
    "license_note": (
        "Combined dataset has no single asserted license; "
        "review original source terms."
    ),
    "limitations": [
        "Positive labels are not confirmed phishing verdicts.",
        "No verified dedicated BEC, spoofing, or malware labels.",
        "Historical data may not represent modern threats.",
        "Initial duplicate counts do not establish leakage absence.",
    ],
}

output_path = Path("mailsleuth_dataset_audit.json")
output_path.write_text(
    json.dumps(audit, indent=2),
    encoding="utf-8",
)

# Verify that the file was written and can be read.
saved_audit = json.loads(output_path.read_text(encoding="utf-8"))

assert saved_audit["dataset_revision"] == (
    "c9440a2fd59c3992e53c361f6afbb4fa8952cd17"
)

print("Audit artifact created")
print("Path:", output_path.resolve())
print("Dataset revision:", saved_audit["dataset_revision"])
print("Training rows:", saved_audit["train_rows"])
print("Test rows:", saved_audit["test_rows"])


Audit artifact created
Path: /content/mailsleuth_dataset_audit.json
Dataset revision: c9440a2fd59c3992e53c361f6afbb4fa8952cd17
Training rows: 162413
Test rows: 40604


In [16]:

%pip install -q "datasets==3.6.0" pandas scikit-learn matplotlib


In [17]:

import sys
import pandas as pd
import sklearn
import datasets

print("Python:", sys.version)
print("Pandas:", pd.__version__)
print("Scikit-learn:", sklearn.__version__)
print("Hugging Face Datasets:", datasets.__version__)
print("Environment setup successful")


Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
Pandas: 2.2.3
Scikit-learn: 1.6.1
Hugging Face Datasets: 4.8.5
Environment setup successful


In [18]:

from datasets import load_dataset

DATASET_ID = "mamtakumar/seven-phishing-email-datasets"
DATASET_REVISION = "c9440a2fd59c3992e53c361f6afbb4fa8952cd17"

dataset = load_dataset(
    DATASET_ID,
    revision=DATASET_REVISION
)

print(dataset)


DatasetDict({
    train: Dataset({
        features: ['text', 'subject', 'label', 'sender', 'receiver', 'date', 'urls', 'dataset_name'],
        num_rows: 162413
    })
    test: Dataset({
        features: ['text', 'subject', 'label', 'sender', 'receiver', 'date', 'urls', 'dataset_name'],
        num_rows: 40604
    })
})


In [19]:

for split_name, split_data in dataset.items():
    print(f"\n--- {split_name.upper()} ---")
    print("Rows:", len(split_data))
    print("Columns:", split_data.column_names)
    print("Features:", split_data.features)

    print("\nFirst record:")
    print(split_data[0])



--- TRAIN ---
Rows: 162413
Columns: ['text', 'subject', 'label', 'sender', 'receiver', 'date', 'urls', 'dataset_name']
Features: {'text': Value('string'), 'subject': Value('string'), 'label': Value('int64'), 'sender': Value('string'), 'receiver': Value('string'), 'date': Value('timestamp[ns]'), 'urls': Value('int64'), 'dataset_name': Value('string')}

First record:
{'text': "\n \n \n\n\nmeat helpful round An afterclang of Cowley's mouth chords closed, died on tdesire And Richie like debt Goulding drank his noise Power and Leopold Blsuccessful struck cough Libel action, says he, slow for ten thousand pounds.Very umbrella tame kind hat grip of you, says Bloom.\nQuavering the match haunt cart chords strayed hope from the air, found iawoken bite drip Foreign wars is the along cause of it.damage taught That's right, net Father Cowley plant said. The reverendI have disgust well only conceived sold a play for the mummers, he saiincrease Shut the book support quick. sold answer Don't let see.

In [20]:

import pandas as pd

for split_name, split_data in dataset.items():
    df = split_data.to_pandas()

    # Treat missing values and whitespace-only strings separately.
    text = df["text"].fillna("").astype(str)
    blank_mask = text.str.strip().eq("")

    print(f"\n{'=' * 50}")
    print(f"SPLIT: {split_name.upper()}")
    print(f"{'=' * 50}")

    print("\nLabel counts:")
    print(df["label"].value_counts(dropna=False).sort_index())

    print("\nLabel proportions:")
    print(
        df["label"].value_counts(normalize=True, dropna=False)
        .sort_index()
        .round(4)
    )

    print("\nMissing text values:", df["text"].isna().sum())
    print("Blank text values:", blank_mask.sum())

    print("\nText length statistics (characters):")
    print(text.str.len().describe())

    print("\nRows by original dataset:")
    print(df["dataset_name"].value_counts(dropna=False))

    print("\nMissing values by column:")
    print(df.isna().sum())

    print("\nShortest nonblank text examples:")
    lengths = text.str.len()
    valid_indices = df.index[~blank_mask]
    shortest_indices = lengths.loc[valid_indices].nsmallest(3).index

    for idx in shortest_indices:
        print({
            "label": df.loc[idx, "label"],
            "dataset_name": df.loc[idx, "dataset_name"],
            "subject": str(df.loc[idx, "subject"])[:120],
            "text_preview": text.loc[idx][:200],
        })



SPLIT: TRAIN

Label counts:
label
0    86951
1    75462
Name: count, dtype: int64

Label proportions:
label
0    0.5354
1    0.4646
Name: proportion, dtype: float64

Missing text values: 0
Blank text values: 4

Text length statistics (characters):
count    1.624130e+05
mean     1.769954e+03
std      1.101470e+04
min      0.000000e+00
25%      3.440000e+02
50%      7.760000e+02
75%      1.752000e+03
max      2.550619e+06
Name: text, dtype: float64

Rows by original dataset:
dataset_name
TREC-05     44461
TREC-07     42964
CEAS-08     31150
Enron       23849
TREC-06     13113
Assassin     4585
Ling         2291
Name: count, dtype: int64

Missing values by column:
text                0
subject             0
label               0
sender          26140
receiver        26140
date            28898
urls            26140
dataset_name        0
dtype: int64

Shortest nonblank text examples:
{'label': np.int64(1), 'dataset_name': 'TREC-05', 'subject': '�����\u2fe1 �ٷ� �����ϴ� ������ -������ �����

In [21]:

import pandas as pd

train_df = dataset["train"].to_pandas()
test_df = dataset["test"].to_pandas()

def audit_text_quality(df, split_name):
    text = df["text"].fillna("").astype(str)
    stripped = text.str.strip()
    lengths = stripped.str.len()

    print(f"\n{'=' * 45}")
    print(split_name)
    print(f"{'=' * 45}")

    print("Empty bodies:", (lengths == 0).sum())
    print("Bodies under 10 characters:", (lengths < 10).sum())
    print("Bodies under 50 characters:", (lengths < 50).sum())
    print("Bodies over 100,000 characters:", (lengths > 100_000).sum())

    print("Rows containing replacement character:",
          text.str.contains("\uFFFD", regex=False).sum())

    print("Exact duplicate bodies after trimming:",
          stripped.duplicated(keep=False).sum())

    print("\nShort-body label distribution (under 10 chars):")
    print(df.loc[lengths < 10, "label"].value_counts().sort_index())

    print("\nShort-body examples:")
    cols = ["label", "dataset_name", "subject", "text"]
    short_df = df.loc[lengths < 10, cols].head(10)

    for _, row in short_df.iterrows():
        print({
            "label": row["label"],
            "dataset": row["dataset_name"],
            "subject": str(row["subject"])[:80],
            "text": str(row["text"])[:80],
        })

audit_text_quality(train_df, "TRAIN")
audit_text_quality(test_df, "TEST")

# Compare body hashes to identify potential train/test overlap.
# Whitespace at the beginning and end is ignored.
train_bodies = train_df["text"].fillna("").astype(str).str.strip()
test_bodies = test_df["text"].fillna("").astype(str).str.strip()

train_hashes = set(
    pd.util.hash_pandas_object(train_bodies, index=False).tolist()
)
test_hashes = set(
    pd.util.hash_pandas_object(test_bodies, index=False).tolist()
)

overlap_hashes = train_hashes.intersection(test_hashes)

print("\n" + "=" * 45)
print("TRAIN/TEST OVERLAP")
print("=" * 45)
print("Unique train body hashes:", len(train_hashes))
print("Unique test body hashes:", len(test_hashes))
print("Shared body hashes:", len(overlap_hashes))



TRAIN
Empty bodies: 4
Bodies under 10 characters: 179
Bodies under 50 characters: 2336
Bodies over 100,000 characters: 65
Rows containing replacement character: 6439
Exact duplicate bodies after trimming: 1574

Short-body label distribution (under 10 chars):
label
0     27
1    152
Name: count, dtype: int64

Short-body examples:
{'label': 0, 'dataset': 'Enron', 'subject': 'elena chilkina', 'text': 'hi'}
{'label': 1, 'dataset': 'TREC-05', 'subject': 'ī��볳! �����δ���! ���ڱݴ���! (100%�� �Ｎ�����ص帳�ϴ�.-��ä�ƴ�!)', 'text': 'MOON2'}
{'label': 0, 'dataset': 'Enron', 'subject': 'delainey and i started initial discussions reguarding ees and the customers and ', 'text': 'john'}
{'label': 1, 'dataset': 'TREC-05', 'subject': '', 'text': 'test'}
{'label': 1, 'dataset': 'TREC-05', 'subject': '���H�Υd���H�Ъ`�N!!!!!', 'text': '���R�W -1'}
{'label': 1, 'dataset': 'TREC-05', 'subject': '������ ���� �ʸ����� �鵵�� 12,500���� �帳�ϴ�. (���� ������)', 'text': ''}
{'label': 1, 'dataset': 'TREC-05', 'subject': '1

In [22]:

import pandas as pd

def prepare_body_summary(df, split_name):
    temp = pd.DataFrame({
        "body": df["text"].fillna("").astype(str).str.strip(),
        "label": df["label"],
    })

    return (
        temp.groupby("body", dropna=False)
        .agg(
            row_count=("label", "size"),
            labels=("label", lambda s: tuple(sorted(s.unique()))),
        )
        .reset_index()
        .rename(columns={
            "row_count": f"{split_name}_rows",
            "labels": f"{split_name}_labels",
        })
    )

train_summary = prepare_body_summary(train_df, "train")
test_summary = prepare_body_summary(test_df, "test")

overlap = train_summary.merge(
    test_summary,
    on="body",
    how="inner",
    validate="one_to_one",
)

print("Shared distinct bodies:", len(overlap))
print("Training rows represented in overlap:",
      overlap["train_rows"].sum())
print("Test rows represented in overlap:",
      overlap["test_rows"].sum())

# Check whether labels are consistent across the two splits.
overlap["labels_agree"] = overlap.apply(
    lambda row: set(row["train_labels"]) == set(row["test_labels"]),
    axis=1,
)

print("\nShared bodies with matching label sets:",
      overlap["labels_agree"].sum())
print("Shared bodies with different label sets:",
      (~overlap["labels_agree"]).sum())

print("\nExamples with different label sets:")
print(
    overlap.loc[
        ~overlap["labels_agree"],
        ["body", "train_rows", "train_labels",
         "test_rows", "test_labels"],
    ].head(10).to_string(index=False)
)

print("\nMost repeated shared bodies:")
print(
    overlap.assign(
        total_rows=overlap["train_rows"] + overlap["test_rows"]
    )
    .sort_values("total_rows", ascending=False)
    [["train_rows", "train_labels", "test_rows",
      "test_labels", "body"]]
    .head(10)
    .to_string(index=False)
)


Shared distinct bodies: 323
Training rows represented in overlap: 554
Test rows represented in overlap: 380

Shared bodies with matching label sets: 323
Shared bodies with different label sets: 0

Examples with different label sets:
Empty DataFrame
Columns: [body, train_rows, train_labels, test_rows, test_labels]
Index: []

Most repeated shared bodies:
 train_rows train_labels  test_rows test_labels                                                                                                                                                   body
         10         (0,)          7        (0,)                                                                                Please review the attached outage report.\n\nJerry Graves\n713-345-8923
         14         (0,)          1        (0,)                                                                                                            Sharon Brown\nETS Gas Control\n713-853-5660
         12         (1,)          2        (1,) 

In [23]:

import pandas as pd

def normalize_column(series):
    return (
        series.fillna("")
        .astype(str)
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

def prepare_email_keys(df, split_name):
    result = pd.DataFrame({
        "body": normalize_column(df["text"]),
        "subject": normalize_column(df["subject"]),
        "label": df["label"].astype(int),
    })

    result["body_key"] = result["body"]
    result["email_key"] = result["subject"] + "\n" + result["body"]

    print(f"\n{split_name}")
    print("Rows with empty body:", result["body_key"].eq("").sum())

    body_label_counts = result.groupby("body_key")["label"].nunique()
    email_label_counts = result.groupby("email_key")["label"].nunique()

    print("Distinct bodies with conflicting labels:",
          (body_label_counts > 1).sum())
    print("Distinct subject+body keys with conflicting labels:",
          (email_label_counts > 1).sum())

    return result

train_keys = prepare_email_keys(train_df, "TRAIN")
test_keys = prepare_email_keys(test_df, "TEST")

shared_bodies = set(train_keys["body_key"]) & set(test_keys["body_key"])
shared_emails = set(train_keys["email_key"]) & set(test_keys["email_key"])

print("\nCROSS-SPLIT COMPARISON")
print("Shared distinct body keys:", len(shared_bodies))
print("Shared distinct subject+body keys:", len(shared_emails))

# Check whether shared bodies also have the same subject.
train_pairs = set(zip(train_keys["subject"], train_keys["body_key"]))
test_pairs = set(zip(test_keys["subject"], test_keys["body_key"]))

shared_pairs = train_pairs & test_pairs

print("Shared exact subject+body pairs:", len(shared_pairs))
print("Shared bodies without an exact subject+body match:",
      len(shared_bodies) - len({
          body for subject, body in shared_pairs
      }))

# Inspect a few body overlaps where subjects differ.
train_subjects = train_keys.groupby("body_key")["subject"].agg(
    lambda s: sorted(set(s))
)
test_subjects = test_keys.groupby("body_key")["subject"].agg(
    lambda s: sorted(set(s))
)

different_subject_examples = []

for body in shared_bodies:
    train_subs = set(train_subjects.get(body, []))
    test_subs = set(test_subjects.get(body, []))

    if train_subs and test_subs and not (train_subs & test_subs):
        different_subject_examples.append({
            "body_preview": body[:100],
            "train_subjects": list(train_subs)[:3],
            "test_subjects": list(test_subs)[:3],
        })

print("\nShared bodies with no common subject:")
for example in different_subject_examples[:5]:
    print(example)



TRAIN
Rows with empty body: 4
Distinct bodies with conflicting labels: 0
Distinct subject+body keys with conflicting labels: 0

TEST
Rows with empty body: 0
Distinct bodies with conflicting labels: 0
Distinct subject+body keys with conflicting labels: 0

CROSS-SPLIT COMPARISON
Shared distinct body keys: 528
Shared distinct subject+body keys: 248
Shared exact subject+body pairs: 248
Shared bodies without an exact subject+body match: 281

Shared bodies with no common subject:
{'body_preview': 'Hi to all And HAPPY NEW YEAR. I have a question. How can i convert a floating variable into a intege', 'train_subjects': ['Type conversion IC.'], 'test_subjects': ['type conversion']}
{'body_preview': 'You have been requested to take part in this poll http://www.donecan.com', 'train_subjects': ['Satisfy your partner every night'], 'test_subjects': ['Answers to erectile dysfunction']}
{'body_preview': 'Women with menopause endorses product for men http://www.fiftywait.com/', 'train_subjects': ['Hel

In [24]:

import pandas as pd

def normalize(series):
    return (
        series.fillna("")
        .astype(str)
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

train_body = normalize(train_df["text"])
test_body = normalize(test_df["text"])

train_subject = normalize(train_df["subject"])
test_subject = normalize(test_df["subject"])

# Do not use empty bodies as duplicate keys.
test_nonempty_bodies = set(test_body[test_body.ne("")])
test_nonempty_pairs = set(
    zip(
        test_subject[test_body.ne("")],
        test_body[test_body.ne("")],
    )
)

train_body_overlap = (
    train_body.ne("") & train_body.isin(test_nonempty_bodies)
)

train_pair_overlap = pd.Series(
    [
        (subject, body) in test_nonempty_pairs if body else False
        for subject, body in zip(train_subject, train_body)
    ],
    index=train_df.index,
)

print("TRAINING ROW IMPACT")
print("Total training rows:", len(train_df))
print("Rows whose nonempty body appears in test:",
      int(train_body_overlap.sum()))
print("Rows whose nonempty subject+body appears in test:",
      int(train_pair_overlap.sum()))
print("Training rows remaining after body-overlap filtering:",
      int((~train_body_overlap).sum()))

print("\nTEST SET IMPACT")
print("Total test rows:", len(test_df))
print("Rows with a nonempty body also present in train:",
      int((test_body.ne("") & test_body.isin(set(train_body[train_body.ne("")]))).sum()))

print("\nLABELS OF TRAINING ROWS WITH BODY OVERLAP")
print(train_df.loc[train_body_overlap, "label"].value_counts().sort_index())

print("\nOVERLAP BY ORIGINAL DATASET")
print(
    train_df.loc[train_body_overlap, "dataset_name"]
    .value_counts()
)


TRAINING ROW IMPACT
Total training rows: 162413
Rows whose nonempty body appears in test: 1058
Rows whose nonempty subject+body appears in test: 499
Training rows remaining after body-overlap filtering: 161355

TEST SET IMPACT
Total test rows: 40604
Rows with a nonempty body also present in train: 653

LABELS OF TRAINING ROWS WITH BODY OVERLAP
label
0    251
1    807
Name: count, dtype: int64

OVERLAP BY ORIGINAL DATASET
dataset_name
CEAS-08     388
TREC-07     299
TREC-05     195
Enron       129
TREC-06      36
Assassin     11
Name: count, dtype: int64


In [25]:

import pandas as pd

def normalize(series):
    return (
        series.fillna("")
        .astype(str)
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

train_check = pd.DataFrame({
    "body": normalize(train_df["text"]),
    "label": train_df["label"].astype(int),
})

test_check = pd.DataFrame({
    "body": normalize(test_df["text"]),
    "label": test_df["label"].astype(int),
})

# Empty bodies are not useful duplicate keys.
train_check = train_check[train_check["body"].ne("")]
test_check = test_check[test_check["body"].ne("")]

train_labels_by_body = train_check.groupby("body")["label"].agg(
    lambda labels: tuple(sorted(labels.unique()))
)
test_labels_by_body = test_check.groupby("body")["label"].agg(
    lambda labels: tuple(sorted(labels.unique()))
)

shared_bodies = set(train_labels_by_body.index) & set(
    test_labels_by_body.index
)

conflicts = [
    {
        "body_preview": body[:150],
        "train_labels": train_labels_by_body[body],
        "test_labels": test_labels_by_body[body],
    }
    for body in shared_bodies
    if train_labels_by_body[body] != test_labels_by_body[body]
]

print("Shared nonempty normalized bodies:", len(shared_bodies))
print("Shared bodies with conflicting label sets:", len(conflicts))

for item in conflicts[:10]:
    print(item)


Shared nonempty normalized bodies: 528
Shared bodies with conflicting label sets: 0


In [26]:

import pandas as pd

def normalize_text(series):
    return (
        series.fillna("")
        .astype(str)
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

# Preserve original data.
train_original = train_df.copy()
test_original = test_df.copy()

train_body = normalize_text(train_original["text"])
test_body = normalize_text(test_original["text"])

# Ignore empty bodies when checking overlap.
test_bodies = set(test_body[test_body.ne("")])

overlap_mask = train_body.ne("") & train_body.isin(test_bodies)

# Create a separate training view; do not modify the original.
train_clean = train_original.loc[~overlap_mask].copy()

print("Original training rows:", len(train_original))
print("Training rows excluded for test-body overlap:", int(overlap_mask.sum()))
print("Leakage-controlled training rows:", len(train_clean))
print("Original test rows:", len(test_original))

# Verify that no nonempty normalized body remains shared.
remaining_train_bodies = set(
    normalize_text(train_clean["text"]).loc[
        normalize_text(train_clean["text"]).ne("")
    ]
)
remaining_test_bodies = set(
    test_body.loc[test_body.ne("")]
)

remaining_overlap = remaining_train_bodies & remaining_test_bodies

print("Remaining shared nonempty body keys:", len(remaining_overlap))

assert len(remaining_overlap) == 0
assert len(train_clean) + int(overlap_mask.sum()) == len(train_original)

print("Verification passed.")


Original training rows: 162413
Training rows excluded for test-body overlap: 1058
Leakage-controlled training rows: 161355
Original test rows: 40604
Remaining shared nonempty body keys: 0
Verification passed.


In [27]:

import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

def normalize_text(series):
    return (
        series.fillna("")
        .astype(str)
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

# Work on a separate copy.
model_data = train_clean.copy().reset_index(drop=True)

# Keep the original subject and body columns unchanged.
model_data["subject"] = model_data["subject"].fillna("").astype(str)
model_data["text"] = model_data["text"].fillna("").astype(str)

# Prepare labels.
y = model_data["label"].astype(int)

# Group identical nonempty bodies together.
# For empty bodies, use the subject as the grouping key.
body = normalize_text(model_data["text"])
subject = normalize_text(model_data["subject"])

groups = body.copy()
empty_body = body.eq("")

groups.loc[~empty_body] = "BODY:" + body.loc[~empty_body]
groups.loc[empty_body & subject.ne("")] = (
    "EMPTY_BODY_SUBJECT:" + subject.loc[empty_body & subject.ne("")]
)

# If both subject and body are empty, keep the row in its own group.
empty_both = empty_body & subject.eq("")
groups.loc[empty_both] = [
    f"EMPTY_ROW:{i}" for i in model_data.index[empty_both]
]

# Create one 80/20 stratified, group-aware split.
splitter = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

train_idx, val_idx = next(
    splitter.split(model_data, y, groups=groups)
)

train_part = model_data.iloc[train_idx].copy()
val_part = model_data.iloc[val_idx].copy()

train_groups = set(groups.iloc[train_idx])
val_groups = set(groups.iloc[val_idx])

print("Training rows:", len(train_part))
print("Validation rows:", len(val_part))
print("Training proportion:", round(len(train_part) / len(model_data), 4))
print("Validation proportion:", round(len(val_part) / len(model_data), 4))

print("\nTraining label proportions:")
print(train_part["label"].value_counts(normalize=True).sort_index().round(4))

print("\nValidation label proportions:")
print(val_part["label"].value_counts(normalize=True).sort_index().round(4))

print("\nShared body groups across training and validation:",
      len(train_groups & val_groups))

assert len(train_groups & val_groups) == 0
assert set(train_part.index).isdisjoint(set(val_part.index))

print("\nGroup-aware split verification passed.")


Training rows: 129110
Validation rows: 32245
Training proportion: 0.8002
Validation proportion: 0.1998

Training label proportions:
label
0    0.5378
1    0.4622
Name: proportion, dtype: float64

Validation label proportions:
label
0    0.5354
1    0.4646
Name: proportion, dtype: float64

Shared body groups across training and validation: 0

Group-aware split verification passed.


In [28]:

import pandas as pd

def build_email_text(df):
    subject = df["subject"].fillna("").astype(str).str.strip()
    body = df["text"].fillna("").astype(str).str.strip()

    return (
        "Subject: " + subject + "\n"
        + "Body: " + body
    )

X_train = build_email_text(train_part)
X_val = build_email_text(val_part)

y_train = train_part["label"].astype(int)
y_val = val_part["label"].astype(int)

print("Training documents:", len(X_train))
print("Validation documents:", len(X_val))

print("\nExample model input:")
print(X_train.iloc[0][:1000])

print("\nTraining label counts:")
print(y_train.value_counts().sort_index())

print("\nValidation label counts:")
print(y_val.value_counts().sort_index())

assert X_train.notna().all()
assert X_val.notna().all()
assert len(X_train) == len(y_train)
assert len(X_val) == len(y_val)

print("\nText preparation verification passed.")


Training documents: 129110
Validation documents: 32245

Example model input:
Subject: When will it be okay
Body: meat helpful round An afterclang of Cowley's mouth chords closed, died on tdesire And Richie like debt Goulding drank his noise Power and Leopold Blsuccessful struck cough Libel action, says he, slow for ten thousand pounds.Very umbrella tame kind hat grip of you, says Bloom.
Quavering the match haunt cart chords strayed hope from the air, found iawoken bite drip Foreign wars is the along cause of it.damage taught That's right, net Father Cowley plant said. The reverendI have disgust well only conceived sold a play for the mummers, he saiincrease Shut the book support quick. sold answer Don't let see.
cow whistle left The strangers, says the decision citizen. Our own fault.marry Go structure swift scrape on! Do! Sonnez!pain doubt behind O ring hell! says I.Bending, she attack list nipped spell a peak ornithic of skirt above her knee
And says Joe, sticking end his increase fi

In [29]:

import time
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
)

baseline = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            lowercase=True,
            strip_accents="unicode",
            ngram_range=(1, 2),
            min_df=2,
            max_df=0.98,
            max_features=100_000,
            sublinear_tf=True,
            dtype=np.float32,
        ),
    ),
    (
        "classifier",
        LogisticRegression(
            C=1.0,
            max_iter=1000,
            random_state=42,
        ),
    ),
])

print("Training TF-IDF + Logistic Regression...")
start = time.time()

baseline.fit(X_train, y_train)

print(f"Training completed in {(time.time() - start) / 60:.2f} minutes")

# Validation predictions only. The held-out test set remains untouched.
val_pred = baseline.predict(X_val)
val_prob = baseline.predict_proba(X_val)[:, 1]

print("\nVALIDATION RESULTS")
print("Accuracy:", round(accuracy_score(y_val, val_pred), 4))
print("ROC-AUC:", round(roc_auc_score(y_val, val_prob), 4))

print("\nClassification report:")
print(
    classification_report(
        y_val,
        val_pred,
        labels=[0, 1],
        target_names=["Label 0: benign/legitimate", "Label 1: phishing/spam"],
        digits=4,
        zero_division=0,
    )
)

print("Confusion matrix (rows=actual, columns=predicted):")
print(confusion_matrix(y_val, val_pred, labels=[0, 1]))

print("\nNumber of TF-IDF features:",
      len(baseline.named_steps["tfidf"].vocabulary_))


Training TF-IDF + Logistic Regression...
Training completed in 2.81 minutes

VALIDATION RESULTS
Accuracy: 0.9816
ROC-AUC: 0.998

Classification report:
                            precision    recall  f1-score   support

Label 0: benign/legitimate     0.9827    0.9830    0.9829     17265
    Label 1: phishing/spam     0.9804    0.9801    0.9802     14980

                  accuracy                         0.9816     32245
                 macro avg     0.9816    0.9815    0.9815     32245
              weighted avg     0.9816    0.9816    0.9816     32245

Confusion matrix (rows=actual, columns=predicted):
[[16971   294]
 [  298 14682]]

Number of TF-IDF features: 100000


In [30]:

import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

# Match predictions to their validation records.
val_results = val_part.reset_index(drop=True).copy()
val_results["actual"] = y_val.to_numpy()
val_results["predicted"] = val_pred
val_results["probability_label_1"] = val_prob
val_results["text_length"] = (
    val_results["text"].fillna("").astype(str).str.len()
)

val_results["error_type"] = "Correct"
val_results.loc[
    (val_results["actual"] == 0) &
    (val_results["predicted"] == 1),
    "error_type"
] = "False positive"

val_results.loc[
    (val_results["actual"] == 1) &
    (val_results["predicted"] == 0),
    "error_type"
] = "False negative"

print("VALIDATION PERFORMANCE BY ORIGINAL DATASET")
rows = []

for source, group in val_results.groupby("dataset_name"):
    rows.append({
        "dataset": source,
        "rows": len(group),
        "accuracy": accuracy_score(
            group["actual"], group["predicted"]
        ),
        "precision_label_1": precision_score(
            group["actual"], group["predicted"],
            pos_label=1, zero_division=0
        ),
        "recall_label_1": recall_score(
            group["actual"], group["predicted"],
            pos_label=1, zero_division=0
        ),
        "f1_label_1": f1_score(
            group["actual"], group["predicted"],
            pos_label=1, zero_division=0
        ),
    })

source_metrics = pd.DataFrame(rows).sort_values("rows", ascending=False)
print(source_metrics.round(4).to_string(index=False))

print("\nERROR COUNTS")
print(val_results["error_type"].value_counts())

# Inspect examples without printing sender or receiver addresses.
for error_type in ["False positive", "False negative"]:
    examples = val_results[val_results["error_type"] == error_type]

    print(f"\n{error_type.upper()} EXAMPLES")
    for _, row in examples.head(5).iterrows():
        print({
            "dataset": row["dataset_name"],
            "actual": int(row["actual"]),
            "predicted": int(row["predicted"]),
            "probability_label_1": round(
                float(row["probability_label_1"]), 4
            ),
            "subject": str(row["subject"])[:160],
            "text_preview": str(row["text"])[:300],
        })


VALIDATION PERFORMANCE BY ORIGINAL DATASET
 dataset  rows  accuracy  precision_label_1  recall_label_1  f1_label_1
 TREC-05  8885    0.9764             0.9690          0.9742      0.9716
 TREC-07  8511    0.9820             0.9945          0.9727      0.9835
 CEAS-08  6159    0.9890             0.9820          0.9982      0.9900
   Enron  4767    0.9832             0.9842          0.9798      0.9820
 TREC-06  2546    0.9808             0.9433          0.9809      0.9617
Assassin   937    0.9723             0.9500          0.9628      0.9564
    Ling   440    0.9864             0.9333          0.9859      0.9589

ERROR COUNTS
error_type
Correct           31653
False negative      298
False positive      294
Name: count, dtype: int64

FALSE POSITIVE EXAMPLES
{'dataset': 'Assassin', 'actual': 0, 'predicted': 1, 'probability_label_1': 0.7867, 'subject': "[zzzzteana] Compensation for World's youngest mother", 'text_preview': "http://www.cnn.com/2002/WORLD/americas/08/26/peru.mother.reut/ind

In [31]:

import pandas as pd

# Reuse val_results from the previous cell.
# The model probability is the predicted probability of label 1.

val_results["confidence"] = (
    val_results["probability_label_1"].where(
        val_results["predicted"] == 1,
        1 - val_results["probability_label_1"],
    )
)

# Errors the model was most confident about.
errors = val_results[
    val_results["actual"] != val_results["predicted"]
].copy()

print("MOST CONFIDENT ERRORS")
for _, row in errors.sort_values(
    "confidence", ascending=False
).head(15).iterrows():
    print({
        "dataset": row["dataset_name"],
        "actual_label": int(row["actual"]),
        "predicted_label": int(row["predicted"]),
        "probability_label_1": round(
            float(row["probability_label_1"]), 4
        ),
        "confidence_in_prediction": round(
            float(row["confidence"]), 4
        ),
        "subject": str(row["subject"])[:160],
        "text_preview": str(row["text"])[:350],
    })

print("\nMOST UNCERTAIN PREDICTIONS")
uncertain = val_results.assign(
    distance_from_half=(
        val_results["probability_label_1"] - 0.5
    ).abs()
).sort_values("distance_from_half")

for _, row in uncertain.head(10).iterrows():
    print({
        "dataset": row["dataset_name"],
        "actual_label": int(row["actual"]),
        "predicted_label": int(row["predicted"]),
        "probability_label_1": round(
            float(row["probability_label_1"]), 4
        ),
        "subject": str(row["subject"])[:160],
        "text_preview": str(row["text"])[:250],
    })


MOST CONFIDENT ERRORS
{'dataset': 'Enron', 'actual_label': 1, 'predicted_label': 0, 'probability_label_1': 0.0019, 'confidence_in_prediction': 0.9981, 'subject': 're : selina', 'text_preview': 'leora\n'}
{'dataset': 'TREC-07', 'actual_label': 0, 'predicted_label': 1, 'probability_label_1': 0.9961, 'confidence_in_prediction': 0.9961, 'subject': '[Reform] cheap oem soft shipping //orldwide', 'text_preview': '\nOEM software: throw packing case, leave CD/DVD, use electronic manuals!\nIf you need software - pay for software ONLY and save 75-90%!\n\nDiscounts! Special offers! For home and office!\n           TOP 1O ITEMS\n\n $49 Windows XP PR0 w/SP2\n $79 MS 0ffice Enterprise 2OO7\n $79 AD0BE Acrobat 8 Pro\n $79 Microsoft Windows Vista Ultimate\n $99 Macromedia Studi'}
{'dataset': 'TREC-07', 'actual_label': 0, 'predicted_label': 1, 'probability_label_1': 0.996, 'confidence_in_prediction': 0.996, 'subject': '[Reform] Three Steps to the Software You Need at the Prices You Want', 'text_preview'

In [32]:

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
)

# Prepare test inputs using exactly the same subject/body format.
X_test = build_email_text(test_original)
y_test = test_original["label"].astype(int)

# The baseline was fitted on train_part, which came from train_clean.
# Do not fit, tune, or select a model using test data.
test_pred = baseline.predict(X_test)
test_prob = baseline.predict_proba(X_test)[:, 1]

print("FINAL TEST RESULTS")
print("Test rows:", len(X_test))
print("Accuracy:", round(accuracy_score(y_test, test_pred), 4))
print("ROC-AUC:", round(roc_auc_score(y_test, test_prob), 4))

print("\nClassification report:")
print(
    classification_report(
        y_test,
        test_pred,
        labels=[0, 1],
        target_names=[
            "Label 0: benign/legitimate",
            "Label 1: phishing/spam",
        ],
        digits=4,
        zero_division=0,
    )
)

print("Confusion matrix (rows=actual, columns=predicted):")
print(confusion_matrix(y_test, test_pred, labels=[0, 1]))


FINAL TEST RESULTS
Test rows: 40604
Accuracy: 0.9812
ROC-AUC: 0.9981

Classification report:
                            precision    recall  f1-score   support

Label 0: benign/legitimate     0.9820    0.9829    0.9824     21738
    Label 1: phishing/spam     0.9803    0.9792    0.9797     18866

                  accuracy                         0.9812     40604
                 macro avg     0.9811    0.9811    0.9811     40604
              weighted avg     0.9812    0.9812    0.9812     40604

Confusion matrix (rows=actual, columns=predicted):
[[21366   372]
 [  392 18474]]


In [33]:
import joblib

MODEL_PATH = "/content/baseline_model.pkl"

joblib.dump(baseline, MODEL_PATH, compress=3)

print("Model saved successfully:", MODEL_PATH)

Model saved successfully: /content/baseline_model.pkl


In [34]:
import json

metrics = {
    "project": "MailSleuth AI",
    "model": "TF-IDF + Logistic Regression",
    "dataset_revision": "c9440a2fd59c3992e53c361f6afbb4fa8952cd17",
    "task": "Binary email text classification",
    "label_mapping": {
        "0": "benign/legitimate according to dataset",
        "1": "phishing/spam according to dataset"
    },
    "validation": {
        "accuracy": 0.9816,
        "roc_auc": 0.9980
    },
    "test": {
        "rows": 40604,
        "accuracy": 0.9812,
        "roc_auc": 0.9981,
        "precision_label_0": 0.9820,
        "recall_label_0": 0.9829,
        "f1_label_0": 0.9824,
        "precision_label_1": 0.9803,
        "recall_label_1": 0.9792,
        "f1_label_1": 0.9797,
        "true_negatives": 21366,
        "false_positives": 372,
        "false_negatives": 392,
        "true_positives": 18474
    },
    "limitations": [
        "Binary dataset labels do not establish verified phishing ground truth",
        "Text-only baseline; no header or attachment forensics",
        "Not yet a multiclass forensic classifier"
    ]
}

with open("/content/baseline_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print("Experiment metrics saved.")

Experiment metrics saved.


In [35]:
import joblib

baseline = joblib.load("models/baseline_model.pkl")

email = """
Subject: Urgent account verification
Body: Click this link to verify your account immediately.
"""

prediction = baseline.predict([email])[0]
probabilities = baseline.predict_proba([email])[0]

print("Predicted dataset class:", prediction)
print("Probability of class 0:", round(probabilities[0], 4))
print("Probability of class 1:", round(probabilities[1], 4))

FileNotFoundError: [Errno 2] No such file or directory: 'models/baseline_model.pkl'

In [36]:
import joblib
import os

MODEL_PATH = "/content/baseline_model.pkl"

print("Model exists:", os.path.exists(MODEL_PATH))

baseline = joblib.load(MODEL_PATH)

print("Model loaded successfully!")

Model exists: True
Model loaded successfully!


In [39]:
email = """
Subject: Urgent account verification
Body: Click this link to verify your account immediately.
"""

prediction = baseline.predict([email])[0]
probabilities = baseline.predict_proba([email])[0]

print("Predicted class:", prediction)
print("Probability of class 0:", round(probabilities[0], 4))
print("Probability of class 1:", round(probabilities[1], 4))

Predicted class: 1
Probability of class 0: 0.005
Probability of class 1: 0.995


In [38]:
from google.colab import files

files.download("/content/baseline_model.pkl")
files.download("/content/baseline_metrics.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>